# Day 9: Streaming and Async

Think about ordering food on Swiggy or Zomato. The total time until the food arrives is the same either way, but compare two apps:

- App A shows a spinner for 35 minutes, then "Delivered".
- App B shows "Order confirmed", then "Preparing", then "Rider picked up", then a live map.

App B *feels* much faster and more trustworthy, even though it isn't. That's what streaming does for AI apps.

An agent that thinks for 20 seconds and then dumps a wall of text feels broken. The same agent showing "Searching... Reading 3 documents... Writing answer..." and then typing the answer word by word feels alive.

Today you'll learn to stream:

- progress after each node
- the LLM's answer, token by token, even from inside a node
- your own custom progress messages
- tool calls from an agent, as they happen

And at the end, **async**, which lets several slow LLM calls run at the same time.

In [ ]:
%pip install -q -U langgraph langchain langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. The stream modes

You've already used two of these on Day 1. Here's the full set you'll actually use:

- `"values"`: the whole state after each step. Good for debugging.
- `"updates"`: just what each node returned. Good for "step X finished" progress.
- `"messages"`: the LLM's text, token by token. Good for chat UIs.
- `"custom"`: anything you send yourself from inside a node. Good for your own progress messages.

To try them, we'll use a small two-step graph: write a story outline, then write the story from it.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class StoryState(TypedDict):
    topic: str
    outline: str
    story: str

In [ ]:
def outline(state: StoryState) -> dict:
    text = llm.invoke(f"Write a 3-point outline for a tiny story about {state['topic']}.").content
    return {"outline": text}

In [ ]:
def write_story(state: StoryState) -> dict:
    text = llm.invoke(f"Write a 5-sentence story from this outline:\n{state['outline']}").content
    return {"story": text}

In [ ]:
builder = StateGraph(StoryState)

In [ ]:
builder.add_sequence([outline, write_story])

In [ ]:
builder.add_edge(START, "outline")
builder.add_edge("write_story", END)

In [ ]:
story_app = builder.compile()

Simple progress with `"updates"`: tell the user each time a step finishes.

In [ ]:
for step in story_app.stream({"topic": "a robot learning to cook biryani"}, stream_mode="updates"):
    node_name = list(step)[0]
    print(f"Finished: {node_name}")

## 2. Streaming the LLM's words

This is the one that surprises people. Our nodes call `llm.invoke()`, which normally waits for the full answer. But when you stream the graph with `stream_mode="messages"`, LangGraph quietly listens to every LLM call inside every node and passes the tokens to you as they arrive. You don't have to change the nodes at all.

Each item you get is a pair: `(token, metadata)`.

### Predict before you run

The graph has two LLM calls (outline, then story). When we stream `"messages"` and print every token, will we see only the story, or both?

<details>
<summary>Click to see the answer</summary>

Both. You'll see the outline being typed out first, then the story, all in one stream. Every LLM call in every node gets streamed.

That's often not what you want in a real app. Next section shows how to keep only one.
</details>

In [ ]:
for token, metadata in story_app.stream({"topic": "a lazy cat"}, stream_mode="messages"):
    print(token.content, end="", flush=True)
print()

## 3. Keeping tokens from one node only

The `metadata` tells you which node each token came from, in `metadata["langgraph_node"]`. So to show only the final story (not the outline), just filter.

In [ ]:
for token, metadata in story_app.stream({"topic": "a brave little auto-rickshaw"}, stream_mode="messages"):
    if metadata["langgraph_node"] == "write_story":
        print(token.content, end="", flush=True)
print()

### Your turn

Change the filter so it shows only the **outline** instead.

In [ ]:
for token, metadata in story_app.stream({"topic": "a monsoon picnic"}, stream_mode="messages"):
    if metadata["langgraph_node"] == "write_story":       # change this to "outline"
        print(token.content, end="", flush=True)
print()

## 4. Sending your own progress messages

Sometimes a node does slow work that isn't an LLM call: reading files, calling an API, querying a database. You'd like to tell the user what's happening.

Inside any node, `get_stream_writer()` gives you a `writer` function. Whatever you pass to it shows up when someone streams with `stream_mode="custom"`. It's like the delivery app's status updates: you decide what to say and when.

In [ ]:
import time
from langgraph.config import get_stream_writer

In [ ]:
class ReportState(TypedDict):
    files: list[str]
    report: str

In [ ]:
def process_files(state: ReportState) -> dict:
    writer = get_stream_writer()
    total = len(state["files"])

    for number, filename in enumerate(state["files"], start=1):
        writer({"progress": f"Reading {filename} ({number} of {total})"})
        time.sleep(0.3)               # pretend this takes a while

    writer({"progress": "Writing the report"})
    return {"report": f"Processed {total} files"}

In [ ]:
builder = StateGraph(ReportState)

In [ ]:
builder.add_node("process_files", process_files)

In [ ]:
builder.add_edge(START, "process_files")
builder.add_edge("process_files", END)

In [ ]:
report_app = builder.compile()

In [ ]:
for event in report_app.stream({"files": ["sales.csv", "users.csv", "orders.csv"]}, stream_mode="custom"):
    print(event["progress"])

## 5. Several modes at once

You can ask for more than one mode by passing a list. Then each item comes as a pair: `(mode, data)`, so you can handle each kind differently.

### Predict before you run

With two files, the node sends three custom messages (two "Reading..." and one "Writing..."), and then finishes. How many lines will be printed, and which one comes last?

<details>
<summary>Click to see the answer</summary>

Four lines. Three `custom` ones while the node is working, then one `updates` line at the very end, when the node returns its result. Custom messages arrive *during* a node; updates arrive *after* it.
</details>

In [ ]:
for mode, data in report_app.stream({"files": ["a.txt", "b.txt"]}, stream_mode=["custom", "updates"]):
    print(f"[{mode}]", data)

## 6. Watching an agent think

For agents, the most useful thing to show the user is which tools are being called. Let's stream the updates from a small stock-price agent and print a friendly line for each event.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

In [ ]:
@tool
def get_stock_price(symbol: str) -> str:
    """Get the latest stock price for an NSE symbol (demo data)."""
    prices = {"TCS": 4120.5, "INFY": 1890.0, "RELIANCE": 2950.25}
    price = prices.get(symbol.upper(), "unknown")
    return f"{symbol.upper()}: Rs. {price}"

In [ ]:
stock_agent = create_agent(
    model=llm,
    tools=[get_stock_price],
    system_prompt="Answer stock questions using the tool.",
)

In [ ]:
question = {"messages": [("user", "Compare the prices of TCS and INFY")]}

for step in stock_agent.stream(question, stream_mode="updates"):
    for node_name, update in step.items():
        for message in update.get("messages", []):
            if getattr(message, "tool_calls", None):
                for call in message.tool_calls:
                    print(f"Calling {call['name']} with {call['args']}")
            elif message.type == "tool":
                print(f"Got back: {message.content}")
            else:
                print(f"Answer: {message.content}")

## 7. Async: doing several slow things at once

When your node calls an LLM, your program mostly just... waits. It's waiting on the network for OpenAI to respond.

Think of making tea and toast. You don't stand and watch the kettle until it boils, then start the toast. You put the kettle on, put the toast in, and both happen at the same time. That's what async is about: while one call is waiting, others can start.

In Python, async functions are written with `async def`, and you `await` the slow parts. Every LangGraph and LangChain method has an async twin: `ainvoke`, `astream`, `llm.ainvoke`. In Jupyter you can use `await` directly in a cell.

We'll ask for three fun facts in parallel. With async, it takes roughly as long as **one** call, not three.

In [ ]:
import operator
from typing import Annotated

In [ ]:
class FactsState(TypedDict):
    facts: Annotated[list[str], operator.add]

In [ ]:
def make_fact_node(topic: str):
    # Each node is an async function that awaits the LLM
    async def fact_node(state: FactsState) -> dict:
        reply = await llm.ainvoke(f"One short fun fact about {topic}.")
        return {"facts": [f"{topic}: {reply.content}"]}
    return fact_node

In [ ]:
builder = StateGraph(FactsState)

In [ ]:
for topic in ("space", "oceans", "honeybees"):
    builder.add_node(topic, make_fact_node(topic))

In [ ]:
for topic in ("space", "oceans", "honeybees"):
    builder.add_edge(START, topic)       # all three start together
    builder.add_edge(topic, END)

In [ ]:
facts_app = builder.compile()
show_graph(facts_app)

In [ ]:
start = time.time()
result = await facts_app.ainvoke({"facts": []})
print(f"3 LLM calls in parallel took {time.time() - start:.1f} seconds\n")

for fact in result["facts"]:
    print("-", fact)

`astream` works the same way. Use `async for` instead of `for`.

In [ ]:
async for token, metadata in story_app.astream({"topic": "a rainy day"}, stream_mode="messages"):
    if metadata["langgraph_node"] == "write_story":
        print(token.content, end="", flush=True)
print()

## 8. Common mistakes

**Mistake 1: Unpacking the wrong shape.**
Each stream mode gives you a different shape, and mixing them up is the most common error. `"messages"` gives pairs `(token, metadata)`. `"updates"` and `"values"` give single dictionaries. A list of modes gives `(mode, data)` pairs.

In [ ]:
try:
    for token, metadata in story_app.stream({"topic": "a kite"}, stream_mode="updates"):
        pass
except ValueError as error:
    print("ValueError:", error)

That error ("too many values to unpack" or "not enough values") almost always means your `for` loop doesn't match the stream mode.

**Mistake 2: Using `await` outside an async context.**
`await` works directly in Jupyter cells, but **not** in a normal `.py` script. In a script, wrap your code in a function and run it with `asyncio.run(main())`.

**Mistake 3: Calling `get_stream_writer()` outside a node.**
The writer only exists while a node is running inside a graph. Call it inside the node, not at the top of your file.

**Mistake 4: Mixing sync and async LLM calls in an async node.**
Inside an `async def` node, use `await llm.ainvoke(...)`. If you use `llm.invoke(...)` there, it still works, but it blocks, and you lose the parallel speed-up.

## 9. Practice

**Exercise 1.** Stream `story_app` with `stream_mode="values"` and print only the **list of keys** in the state after each step. You should see the state grow.

**Exercise 2.** Write a node that "downloads" 5 pages and sends a custom event with the percentage done after each page (20, 40, ... 100). Print a simple text progress bar like `[####------] 40%`.

In [ ]:
# Solution 1
for state in story_app.stream({"topic": "a paper boat"}, stream_mode="values"):
    print(list(state))

In [ ]:
# Solution 2
class DownloadState(TypedDict):
    done: bool

In [ ]:
def download(state: DownloadState) -> dict:
    writer = get_stream_writer()
    for page in range(1, 6):
        time.sleep(0.2)
        writer({"percent": page * 20})
    return {"done": True}

In [ ]:
builder = StateGraph(DownloadState)

In [ ]:
builder.add_node("download", download)

In [ ]:
builder.add_edge(START, "download")
builder.add_edge("download", END)

In [ ]:
download_app = builder.compile()

In [ ]:
for event in download_app.stream({}, stream_mode="custom"):
    filled = event["percent"] // 10
    bar = "#" * filled + "-" * (10 - filled)
    print(f"[{bar}] {event['percent']}%")

## Wrapping up

`"updates"` gives you step-by-step progress, `"messages"` gives you the LLM's tokens from inside any node, `"custom"` carries whatever you send with `get_stream_writer()`, and `"values"` shows the whole state. Use `metadata["langgraph_node"]` to keep only the tokens you want.

For speed, async nodes with `await llm.ainvoke(...)` let parallel branches really run at the same time.

That completes the core of the course. You can now build agents with tools, memory, human approval and streaming, which covers a lot of real-world apps. From tomorrow we move into advanced patterns, starting with getting **structured data** out of the LLM, and the classic workflow designs that experienced teams use.